# Exercise 3 – Jupyter Notebook
# Build multi-target neural network prediction models for an order form autocompletion use case
SAP TechEd 2026, Hands-On Workshop: DA262 - Get hands-on with tabular AI- and forecasting-models in SAP HANA Cloud
<br><br>



---

## Understanding the exercise scenario 

In a large variety of business processes, forms (like an order form) require filling a large number of fields in the UI. In data reality, some fields are often left blank when an order is created — for example, the shipping point, payment terms, or plant. Filling these in correctly requires experience and takes time.

In this exercise, you will build a __neural network machine learning model__ using algorithms from the __Predictive Analysis Library (PAL)__ in SAP HANA Cloud, to predict values to autocomplete 8 fields in a Sales Order form. The __PAL multi-target MLP algorithm__ in its latest release (2026 Q3), it also __allows to handle columns (fields), which are sometimes already filled and sometimes have to predicted, thus have a dual role__. Again, in reality a common pattern too.

In this exercise you will
- __Access and explore the Sales Order Data__
- __Fine-tune a pre-trained model with new Sales Order data__
- __Apply the model to predict and autocomplete new Sales Order cases__
- __Measure and review the prediction accuray__


Overview on the exercise tasks

![](./images/ex3_scenario_overview.png)

<br>

## Ex. 3.0 - Connect to your SAP HANA Cloud instance

### Step 0: Establish and check connection


Throughout this exercise, we will be using the __Python Machine Learning client for SAP HANA (hana-ml)__, as your reference to all its functions and capabilities see the [hana-ml documentation](https://help.sap.com/doc/cd94b08fe2e041c2ba778374572ddba9/2026_3_QRC/en-US/change_log.html). The current version released with SAP HANA Cloud 2026 Q3 release is 2.30.
- The python package hana-ml in general allows to script in python, while SQL code is generated on-the-fly and directly passed to a connected SAP HANA database system for execution.
    - It allows to access and prepare data by means of a HANA dataframe, a python object holding a SQL select query. Many methods are provided to be used with the HANA dataframe, changing the SQL select query behind the scenes.
    - As its core, it provides methods to apply AI functions (algorithms from the [Predictive Analysis Library (PAL)](https://help.sap.com/docs/hana-cloud-database/sap-hana-cloud-sap-hana-database-predictive-analysis-library/sap-hana-cloud-sap-hana-database-predictive-analysis-library-pal?locale=en-US&version=LATEST) and [Automated Predictive Library (APL)](https://help.sap.com/docs/apl?locale=en-US&version=LATEST) to the data prepared with a HANA dataframe, designed to apply all the processing within the SAP HANA database.
- The python package is released as a component with every the SAP HANA Client delivery, in addition the latest __hana-ml__-version can always be found at the pypi public repository at https://pypi.org/project/hana-ml/.
<br>

In Python, installed packages require to be __imported__ to the current session, in order to be available for use in python scripts.

In [ ]:
# Importing the Python Machine Learning client library for SAP HANA and get the version
import hana_ml
print(hana_ml.__version__)

Execute the next cell and the referenced script, prepared in the Ex0 Getting Started section to connect to the SAP HANA CLoud database.

In [ ]:
%run "../ex0/ex0_connect2HANACloud.ipynb"

---
## Ex. 3.1 - Access and explore the Sales Order Data

As Sales Order demo data, we are using the [SALT: Sales Autocompletion Linked Business Tables Dataset](https://openreview.net/forum?id=UZbELpkWIr#discussion) the benchmarking dataset, which was released by the SAP RPT team and is based on the SAP sales-order autocompletion (SOAC) use case. The [SAP/SALT · Datasets at Hugging Face](https://huggingface.co/datasets/SAP/SALT) has about ~29 columns and roughly 1.9M rows for training and 400k rows of test data.

The dataset contains typical SAP sales orders. Each order has **27 fields** split into two groups:
- **19 input fields** — these are always known when an order is created .
    -  for example: which product, which customer (sold-to party), which country (billing country/region), which sales org
- **8 target fields** — these are sometimes left blank and need to be predicted,
    - for example: which plant, which shipping point, what payment terms, which sales office, which incoterms, which shipping condition

The following datasets have be pre-loaded to the SAP HANA Cloud instance, schema __DA262_SHARED__
- A pre-trained MT_MLP model (trained on **1.9 million** Sales Order training data rows) in table __SOAC_PRETRAINED_MODEL__
- Sales order test data sets __SOAC_TESTDATA__ (402k rows) and __SOAC_TESTDATA_SUBSET__ (1000 rows)
    - Sales Order rows, the model it has never seen during training, this is how we measure how well the model generalizes to unseen data based on patterns it learned during training.


The following cell also sets up the table name and column references throughout the exercises.

In [ ]:
# Schema and Tables
DATA_SCHEMA        = 'DA262_SHARED'                    # Schema with pre-loaded data
TEST_TABLE         = 'SOAC_TESTDATA'                   # Full test data table
TEST_TABLE_SMALL   = 'SOAC_TESTDATA_SUBSET'            # Subset of the test data table

MODEL_SCHEMA       = 'DA262_SHARED'                    # Schema with pre-loaded data
MODEL_TABLE        = 'SOAC_PRETRAINED_MODEL'           # Table with pre-trained MT_MLP model
TRAIN_LOG_TABLE    = 'SOAC_PRETRAINED_MODEL_TRAINLOG'  # Progress log table from the training task

USER_SCHEMA        = myuser                            # "myuser" is variable containing your connecting userid
FT_TRAIN_TABLE     = 'SOAC_FINETUNING_TRAININGDATA'    #1000 rows, will be sampled from TEST_TABLE;
FT_MODEL_TABLE     = 'SOAC_FINETUNED_MODEL'
FT_TEST_ACCURACY_TABLE = 'SOAC_FINETUNED_TEST_ACCURACY'


# Columns, and columns roles used for training and finetuning the MT_MLP model
KEY_COL      = 'ID'

# 19 always-known input features
INPUT_COLS = [
    'SALESDOCUMENTTYPE', 'SALESORGANIZATION', 'DISTRIBUTIONCHANNEL',
    'BILLINGCOMPANYCODE', 'TRANSACTIONCURRENCY', 'SALESDOCUMENTITEMCATEGORY',
    'PRODUCT', 'SOLDTOPARTY', 'SHIPTOPARTY', 'BILLTOPARTY', 'PAYERPARTY',
    'COUNTRY_SOLDTOPARTY', 'REGION_SOLDTOPARTY',
    'COUNTRY_SHIPTOPARTY',  'REGION_SHIPTOPARTY',
    'COUNTRY_BILLTOPARTY',  'REGION_BILLTOPARTY',
    'COUNTRY_PAYERPARTY',   'REGION_PAYERPARTY',
]

# For the full, 1.9M Sales Order training data and model, 
#  - all 8 target columns "overlap", i.e. they are target and can be feature at the same time (have a dual role)
#  - "Overlapped" means: 
#     - when a value is prsent → PAL uses it as an input clue, like a regular feature value to predict other targets.
#     - when it's value is missing, the model treats it as target and predicts the value 
#       (character missing value have to be replaced with 'PAL_MLP_MASK', integer missing values with 999)
ALL_TARGETS = [
    'SALESOFFICE', 'SALESGROUP', 'CUSTOMERPAYMENTTERMS', 'PLANT',
    'HEADERINCOTERMSCLASSIFICATION', 'ITEMINCOTERMSCLASSIFICATION',
    'SHIPPINGCONDITION', 'SHIPPINGPOINT',
]

print(f'Input features  : {len(INPUT_COLS)}')
print(f'Predicted fields: {len(ALL_TARGETS)}')
print(f'Model table     : {MODEL_SCHEMA}.{MODEL_TABLE}')

---
<br>

### Step 1: Create HANA dataframes, pointing to the prepared tables in SAP HANA Cloud

The data already lives in SAP HANA Cloud — we do not upload anything from this notebook.

When you run the code below, Python does not download the table. Instead it creates a **HANA DataFrame** — a lightweight pointer that says "I want to work with this table in HANA."
- The data stays inside the database the whole time. The HANA DataFrame simply embodies a SQL select statement against the database.
- Only when you call `.collect()` does the data actually move from HANA into Python. For large tables this distinction matters a lot — you can filter, group, and aggregate inside HANA first, and only bring the small result to Python.
- The dataframe output shows `shape: [402855, 28]` — this means 402855 orders and 28 columns (the ID column plus the 27 data fields). Notice the data did not move to Python just from creating the pointer.

In [ ]:
hdf_test = myconn.table(TEST_TABLE, schema=DATA_SCHEMA)

print(hdf_test.select_statement)
print('Test  shape:', hdf_test.shape)
print(f'Columns ({len(hdf_test.columns)}): {hdf_test.columns}')

---
<br>

### Step 2: Explore the Data

Before using the model, it is good practice to understand the data first.

In this section we will look at three things:
- Preview a few rows and see what a real sales order looks like
- Columns statistical summary, like check for nulls, value ranges, and data quality
- Count unique values per field (fields with more unique values are harder to predict)

__Preview some Sales Orders__  

In [ ]:
# Preview first 3 rows — .collect() brings data from HANA to Python
hdf_test.head(3).collect()


<br>

__Review the columns statistical summary__

The code below runs two things:

**`describe()`** — gives a quick table for every column showing: row count, unique value count, nulls, min and max. Run this cell first and check the `nulls` column — it should be `0` for all fields. Any non-zero value would mean missing data that could affect prediction quality.

For example, SALESOFFICE has only **6 unique values** — the model almost always gets it right. SALESGROUP has **407 unique values** — much harder, so accuracy will be lower.

In [ ]:
hdf_test.describe().collect()


<br>

__How does the value count distribution for each target column look like?__

The bar charts below __show the most common values__ for each of the 8 fields the model predicts.

Think of it like a multiple choice question — a field with 6 possible answers (SALESOFFICE) is much easier to get right than a field with 407 possible answers (SALESGROUP). When you look at the accuracy results later in section 3.4, you will see this pattern clearly: fields with fewer unique values score higher.

In [ ]:
from hana_ml.visualizers.eda import EDAVisualizer
for col in ALL_TARGETS:
    top10 = hdf_test.agg([('count', KEY_COL, 'CNT')], group_by=col).sort('CNT', desc=True).head(10)
    fig = EDAVisualizer( enable_plotly=True).bar_plot(data=top10, column=col, aggregation={'CNT': 'max'}, title=col,   width=600, height=400  )


---
## Ex. 3.2 - Train a multi-target classification model for sales order form autocompletion

We seek to build a  multi-target classification model for this use cases, based on the [__Multi-Task MLP__ algorithm, from the SAP HANA Cloud Predictive Analysis Library (PAL)](https://help.sap.com/docs/hana-cloud-database/sap-hana-cloud-sap-hana-database-predictive-analysis-library/multi-task-multilayer-perceptron-imputer?locale=en-US&version=LATEST)  
Let's first understand **What is a Multi-Task MLP?**
- MLP (Multi-Layer Perceptron) is a type of neural network.
- “Multi-Task” means the model allows to predicts multiple target columns in a single model,  all 8 missing fields in one pass — instead of running 8 separate models. This is faster and lets the fields help each other.
- In it's 2026 Q3 release, the algorithm also __allows to handle columns (fields) with a dual role, columns which are sometimes already filled and can be treated as a feature value and sometimes are empty and have to predicted.



### Step 3: Reviewing the pre-trained MT_MLP model

Training a model from scratch on 1.9 million rows takes about **2.5 hours**, e.g. using a Elasitic Compute Node HANA Cloud instance of size btw 128-256 GB.
- We do not need to wait, therefore a model has already been trained and are saved to SAP HANA Cloud as a regular table.
- Loading the model simply means pointing to that table. No training happens here.

| What | Detail |
|---|---|
| Model type | Multi-Task MLP (neural network) |
| Training rows | 1,900,000 |
| Input columns | 19 always-known fields |
| Predicted fields | 8 (all at once) |
| Training time | ~2.5 hours |
| Stored in | `SOAC_MODEL_FULL_27COL_TBL` — a column table in SAP HANA Cloud |

When you run the cell below, you will see **Model table rows: 14,864** — these are
the neural network weights stored as rows in HANA.

In [ ]:
from hana_ml.algorithms.pal.neural_network import MLPMultiTaskClassifier

# Load the pre-trained model — no training happens here.
# We just point clf.model_ to the table where the weights are stored.
clf = MLPMultiTaskClassifier(overlapped_variable=ALL_TARGETS)
clf.model_ = myconn.table(MODEL_TABLE, schema=MODEL_SCHEMA)

print(f'Model loaded from : {MODEL_SCHEMA}.{MODEL_TABLE}')
print(f'Model table rows  : {clf.model_.count():,}')


<br>  

__Reviewing the training log from the full model training task__  

The Training progress log table, stores the __errors between predicted values and target values__ over the iterative training epochs from the inital start to end of the training task. The numbers below confirm the model trained successfully on 1.9 million orders.
- **Total iterations** shows how many training checkpoints were saved 
- **Start error** shows the  error before any learning — high is normal at the start
- **Final error** shows the error after training — much lower means the model learned well.

A large drop from start to finish (e.g. 4.0 → 0.09) confirms the model converged and is ready to use.


In [ ]:
train_log = myconn.table(TRAIN_LOG_TABLE, schema=MODEL_SCHEMA).collect()

print(f'Training rows   : 1,900,000')
print(f'Total iterations: {len(train_log)}')
print(f'Start error     : {train_log["ERROR"].iloc[0]:.4f}')
print(f'Final error     : {train_log["ERROR"].iloc[-1]:.4f}')


---
<br>

### Step 4: Prepare data and model parameter for Fine-Tuning the model with new New Orders

The model you just loaded learned from **1.9 million** past sales orders.
But the real world keeps changing — new customers, new products, new routes.
Fine-tuning lets the model learn new patterns **without forgetting** what it already knows.

> **Simple analogy:**
> A salesperson with 10 years of experience gets 1,000 orders from a brand-new market.
> They learn the new patterns **on top of** their experience — not instead of it.

Create a HANA Dataframe, sampling 1000 rows from the test data set for finetuning the model.

In [ ]:
# Create the HANA dataframe "hdf_ft"
hdf_ft = myconn.table(TEST_TABLE, schema=DATA_SCHEMA).filter('"ID" BETWEEN 1 AND 1000')
hdf_ft.save((myuser, FT_TRAIN_TABLE), force=True)

print(f"Ready: {myconn.table(FT_TRAIN_TABLE, schema=myuser).count():,} new orders saved to {FT_TRAIN_TABLE}")


<br>

__Review the Model Training Parameters used__

The code in the next step uses the **exact same settings** that were used to train the 1.9M model.
- The __only difference__ is __one new__ parameter added for __fine-tuning: `finetune=True`__.
- The parameter `ENABLE_STREAMING=1` is also new with 2026 Q3 version of the algorithm, allowing for much more efficient way of processing the categorial column data, being faster and consuming much fiewer memory resources.

**Neural network shape:**

| Parameter | Value | What it means |
|---|---|---|
| `network_type` | `resnet` | Type of neural network — good at learning complex patterns |
| `embedded_num` | 32 | Each category gets a list of 32 numbers to represent it |
| `residual_num` | 4 | 4 deep layers stacked inside the network |

**Training settings:**

| Parameter | Value | What it means |
|---|---|---|
| `num_epochs` | 100 | Do at most 100 training rounds |
| `batch_size` | 256 | Look at 256 orders at a time |
| `learning_rate` | 0.01 | How big a step to take when updating the model |
| `training_percentage` | 0.9 | Use 90% for training, 10% to check progress |
| `early_stop` | True | Stop automatically when the model stops improving |
| `save_best_model` | True | Keep the best version found during training |
| `random_state` | 42 | Fixed starting point — results are the same every time |
| `training_style` | `stochastic` | Mini-batch training — same method as the 1.9M training |
| `ENABLE_STREAMING` | 1 | Process data in chunks — same method as the 1.9M training |

**New for fine-tuning:**

| Parameter | Value | What it means |
|---|---|---|
| ⭐ `finetune` | `True` | **Load the 1.9M knowledge first — do NOT start from zero** |

> **Why is `finetune=True` the most important parameter?**
> Without it the model ignores everything it learned from 1.9M orders and starts fresh.
> That is called **catastrophic forgetting** — the model only knows the 1,000 new orders.
> With `finetune=True` it builds on top of 1.9M knowledge instead of replacing it.


---
<br>

### Step 5: Now run the fine-tuning training task

The code below starts the fine-tuning. It passes three things to SAP HANA:
-  **1,000 new orders** for the new patterns to learn 
-  **Training parameters** uses the same settings as the 1.9M training (see  above) 
-  **Pre-trained model** loads the 1.9M knowledge before training begins 

**Expected training time: about 2 minutes.**

When it finishes, check the **Start error** in the output — this confirms whether the 1.9M knowledge was loaded correctly.


In [ ]:
# Prepare the HANA dataframes, for the finetuning-training data and the pre-trained model table
hdf_train = myconn.table(FT_TRAIN_TABLE, schema=myuser)
hdf_pre_model=myconn.table(MODEL_TABLE, schema=MODEL_SCHEMA)

print(f"Pre-trained model table {MODEL_SCHEMA}.{MODEL_TABLE}  ({hdf_pre_model.count():,} rows)") 
print()

In [ ]:
# Establish the MLPMultiTaskClassifier object, and run the finetuning fit task

from hana_ml.algorithms.pal.neural_network import MLPMultiTaskClassifier
clf_ft = MLPMultiTaskClassifier(
    overlapped_variable = ALL_TARGETS,  # Specifying the 8 columns, which overlap / shall have the dual role of target and feature
    network_type        = 'resnet',
    embedded_num        = 32,
    residual_num        = 4,
    num_epochs          = 100,
    batch_size          = 256,
    learning_rate       = 0.01,
    training_percentage = 0.9,
    early_stop          = True,
    save_best_model     = True,         # Allows to directly save the finetuned model to new table
    random_state        = 42,
    training_style      = 'stochastic',
    finetune            = True,
)
clf_ft._extend_pal_parameter({'ENABLE_STREAMING': 1})


import time
print("Training started ... (this takes about 2 minutes)")
t0 = time.time()

clf_ft.fit(data=hdf_train,
        pre_model = hdf_pre_model,         # DA262_SHARED.SOAC_PRETRAINED_MODEL
        model_table_name = FT_MODEL_TABLE, # <user-schema>."SOAC_FINETUNED_MODEL", needs to be dropped beforehand
                                           # the latest patch on end of Oct'26 will allow "force=True" to drop the model table if it exists
                                           # otherwise the fit task, breaks with an error, which we want to avoid.
        key='ID',
        #features=                         # Are given by the pre-trained model, must not be specified again
        label=[],                          # target-only columns (not relevant here)
        categorical_variable=INPUT_COLS + ALL_TARGETS)

training_time = time.time() - t0
ft_log = clf_ft.train_log_.collect()

print(clf_ft.runtime)
print()

print(f"Training time         : {training_time:.0f} seconds")
print(f"Fine-tuned model rows : {clf_ft.model_.count():,}  (original had 14,864)")
print(f"Start error           : {ft_log['ERROR'].iloc[0]:.4f}  (good if ~0.18, bad if ~0.80)")
print(f"Final error           : {ft_log['ERROR'].iloc[-1]:.4f}")
print()



In [ ]:
print("MT_MLP parameters, manually extended with ENABLE_STREAMING:")
print(clf_ft.get_fit_parameters())
print()

In [ ]:
# Uncomment, to review the executed SQL block to finetune the model
print(clf_ft.get_fit_execute_statement())

In [ ]:
# The finetune model has been successfully saved
hdf_mymodel = myconn.table(FT_MODEL_TABLE, schema=myuser)
print(f"Fine-tuned model rows : {hdf_mymodel.count():,}  (original had 14,864)")

---
## Ex. 3.3 - Predict missing fields, autocomplete new sales orders
### Step 6: Predict with the finetuned model
Now we put the model to work on real orders! For that
- We sample **4 orders** from the test set and simulate what happens in production
- We __hide the 8 target field values__ by replacing them with a __special marker__ called __`PAL_MLP_MASK`__ for character columns, it would be __`999`__ for integer columns. This tells the model: *"this value is missing — please predict it."*
- The model then looks at the 19 known input fields (product, customer, country, etc.) and predicts all 8 blank fields at once. We already know the true values for these orders, so we can immediately check whether the predictions are correct.
- The result table shows each order, each field, the true value, the predicted value, and whether they match.

At the bottom of the output you will see a summary like `30 / 32 correct (94%)`. This is based on only 4 orders — too few to be a reliable number. Ex. 3.4 runs the same test on all 1,000 orders to get a trustworthy accuracy figure.

In [ ]:
# Prepare the predict dataframe

# Sample 4 orders
from hana_ml.algorithms.pal.preprocessing import Sampling
smp = Sampling(method='simple_random_without_replacement', sampling_size=4)
sample_orders = smp.fit_transform(data=hdf_test)
display(sample_orders.collect())


# Take 4 sample orders, and set the special mask value 'PAL_MLP_MASK' to all 8 overlapped column field value to set them to get predicted
# PAL_MLP_MASK-value tells the model: "this value is missing — please predict it."
import pandas as pd
MASK = 'PAL_MLP_MASK'
masked_targets = [(f"'{MASK}'", col) for col in ALL_TARGETS]

predict_input = sample_orders.select(KEY_COL, *INPUT_COLS, *masked_targets)

print("Review the column value replacement expressions in the SQL Statement: 'PAL_MLP_MASK' as <column name>")
print(predict_input.select_statement)

display(predict_input.to_head('SALESOFFICE').to_head('ID').collect())


<br>  

__Now apply the predict-method to the prepared prediction HANA dataframe__

In [ ]:
# Now, we execute predict from our MLPMultiTaskClassifier finetuning task object "clf_ft"
result        = clf_ft.predict(predict_input, key=KEY_COL).collect()
print(clf_ft.runtime)

In [ ]:
# Then, we can compare predictions of the 8 target columns to the true values, of these 4 sales orders
truth_df = sample_orders.collect().set_index('ID')
preds    = result[result['TARGET'].isin(ALL_TARGETS)]

rows = []
for _, row in preds.iterrows():
    true_val = truth_df.loc[row['ID'], row['TARGET']]
    rows.append({
        'Order'    : row['ID'],
        'Field'    : row['TARGET'],
        'Truth'    : true_val,
        'Predicted': row['SCORE'],
        'Match'    : '✅' if true_val == row['SCORE'] else '❌'
    })

comparison = pd.DataFrame(rows)
display(comparison)

correct = (comparison['Match'] == '✅').sum()
print(f'Result: {correct} / {len(comparison)} correct  ({correct/len(comparison):.0%})')


---
<br>

## Ex. 3.4 - Measure Prediction Accuracy

4 orders gave us a first impression. Now we test on all **1,000 orders the model has never seen before** to get more reliable accuracy numbers.

### Step 7: Review prediction accuracy of the top 3 ranked predictions
The model does not just give one answer — it gives its **top 3 predicted values** ranked by confidence for each target.
- We measure three metrics using the SALESGROUP field as an example (1,000 orders tested):
    - **Top-1 = 90.7%** The model's first guess was correct for 907 out of 1,000 orders.
    - **Top-3 = 99.1%** The correct answer appeared somewhere in the top 3 guesses for 991 out of 1,000 orders. The model got it wrong on first guess 93 times, but the right answer was still in the top 3 for 84 of those 93 times.
    - **MRR = 0.949** MRR (Mean Reciprocal Rank) is a ranking quality statistic, it measures how high up in the list of multi-class predictions the correctly predicted class value is ranked on average:
        - If the correct answer is always rank 1 → MRR = 1.000 (perfect)
        - If the correct answer is always rank 2 → MRR = 0.500
        - If the correct answer is always rank 3 → MRR = 0.333
        - MRR = 0.949 means the correct answer is almost always the model's first guess — very close to perfect.

**Why Top-3 matters in practice:** In a real SAP applications, the user sees a dropdown with the top 3 suggestions and picks the right one. A Top-3 score of 99% means the correct answer is in the dropdown almost every time.



<br>  

This **first code cell** below loads a small helper function from __ex03_accuracy_utils.py__ for better overview.  
You will see `compute_accuracy() ready` as output, meaning it is prepared but not yet run. 

In [ ]:
from ex03_accuracy_utils import setup_accuracy, compute_accuracy, make_truth_union

setup_accuracy(myconn, ALL_TARGETS, DATA_SCHEMA, TEST_TABLE_SMALL)

# truth_union is used by the confidence analysis cell below
truth_union = make_truth_union(ALL_TARGETS, DATA_SCHEMA, TEST_TABLE)

print('compute_accuracy() ready — call it with any HANA table that has ID / TARGET / SCORE / CONFIDENCE columns.')
print()

<br>  

The **next code cells** run the predictionx on all 1,000 orders and calls the utility functions to compute the accuracy table.  

In [ ]:
# Prepare the predict_all HANA dataframe
hdf_test_small = myconn.table(TEST_TABLE_SMALL, schema=DATA_SCHEMA)

# Take 1000 orders, and set the special mask value 'PAL_MLP_MASK' to all 8 overlapped column field value to set them to get predicted
# PAL_MLP_MASK-value tells the model: "this value is missing — please predict it."
import time
import pandas as pd
MASK = 'PAL_MLP_MASK'
masked_targets = [(f"'{MASK}'", col) for col in ALL_TARGETS]

predict_all    = hdf_test_small.select(KEY_COL, *INPUT_COLS, *masked_targets)
print(predict_all.select_statement)

In [ ]:
# Now, we execute predict from our MLPMultiTaskClassifier finetuning task object "clf_ft"
# verbose=True returns ALL candidate values with confidence scores (not just top-1)
# — we need this to measure Top-3 and MRR.
FT_TEST_ACCURACY_TABLE = 'SOAC_FINETUNED_TEST_ACCURACY'
t0 = time.time()
result_test=clf_ft.predict(predict_all, key=KEY_COL, verbose=True)
std_1k_ms = (time.time() - t0) * 1000
#print(clf_ft.runtime)
print()

result_test.save(FT_TEST_ACCURACY_TABLE, force=True)


# Compute Top-1 / Top-3 / MRR using the shared helper defined above.
accuracy_df = compute_accuracy(FT_TEST_ACCURACY_TABLE)

display(accuracy_df)
print(f'\nStandard predict (1,000 orders): {std_1k_ms:.0f} ms')
print('\nTop-1 = first guess correct  |  Top-3 = correct answer in top 3  |  MRR: 1.000 = always top')


---
## Summary

In this exercise you used a machine learning model to automatically predict 8 missing fields on SAP sales orders — entirely inside SAP HANA Cloud, without moving any data to Python.
- Connected to SAP HANA Cloud 
- Loaded 1,000 test orders already stored in HANA 
- Explored field distributions and data quality 
- Loaded a pre-trained model — no retraining needed 
- Predicted 8 missing fields on 4 sample orders 
- Measured Top-1, Top-3 and MRR accuracy across 1,000 orders 


**Three things to remember:**
- **ML inside SAP HANA Cloud**  The model runs where the data lives — no export, no extra infrastructure 
- **Pre-trained model** Training took 2.5 hours once; after that predictions take milliseconds (especially with stateful models)
- **Multi-target model with dual-roal attributes** MT_MLP is a powerful neural network algorithm, allowing to use of overlapping attributes in complex order form autocompletion scenarios

<br><br>

# Appendix - Reference sections

## A.1 - How to accelerate prediciton latency with PAL stateful models (OPTIONAL)

Standard predict reloads the MT_MLP prediction model from table into parsed state on **every single call** — at the costs of additional hundreds of milliseconds.  
__Stateful predict__ loads and parses the model once into HANA scriptserver runtime memory **once** and reuses them — much faster.

Three steps: **Create** state → **Test** speed → **Delete** state.

__Create__  
- `create_model_state()` returns a **State ID** — a unique reference to the model that is now sitting in memory. In the next steps we pass this State ID to the prediction call instead of the model table, which tells HANA: *"the model is already in memory — use it directly."*

In [ ]:
# Create: load model from source table into HANA scriptserver memory once
clf_ft.create_model_state(force=True)
STATE_ID = clf_ft.state.filter('"NAME" = \'STATE_ID\'').collect()["VALUE"].iloc[0]
print(f"Model is now in HANA server memory.")
print(f"State ID: {STATE_ID}")

<br>

__Now predict on a single row__
- without the regular model 

In [ ]:
import time

MASK = "PAL_MLP_MASK"
masked_targets = [(f"'{MASK}'", col) for col in ALL_TARGETS]
one_order = hdf_test.head(1).select(KEY_COL, *INPUT_COLS, *masked_targets)

# Test standard predict — model reloads from disk each call
t0 = time.time()
clf_ft.predict(one_order, key=KEY_COL).collect()
std_ms = (time.time() - t0) * 1000
print(f"Standard predict (1 order): {std_ms:.0f} ms  — reloads from disk")

<br>

- and with the __stateful model__ for a runtime comparison

In [ ]:
# Test stateful predict — model is already in HANA server memory
MASK = "PAL_MLP_MASK"
mlp_input_cols = [(f"'{MASK}'", col + "_MLP_INPUT") for col in ALL_TARGETS]
sf_one = hdf_test.head(1).select(KEY_COL, *INPUT_COLS, *mlp_input_cols)
sf_one.save("SOAC_SF_ONE", force=True)

# the model table is passed as an empty table with stateful predictions
if myconn.has_table("SOAC_SF_EMPTY_MODEL"):
    myconn.drop_table("SOAC_SF_EMPTY_MODEL")
myconn.execute_sql(
    'CREATE COLUMN TABLE "SOAC_SF_EMPTY_MODEL" '
    '("ROW_INDEX" INTEGER, "PART_INDEX" INTEGER, "MODEL_CONTENT" NVARCHAR(5000))'
)

sql = (
    "DO BEGIN\n"
    "    DECLARE p TABLE (\"PARAM_NAME\" NVARCHAR(256), \"INT_VALUE\" INTEGER,\n"
    "                     \"DOUBLE_VALUE\" DOUBLE, \"STRING_VALUE\" NVARCHAR(1000));\n"
    f"    :p.INSERT(('STATE_ID',      NULL, NULL, '{{STATE_ID}}'));\n"
    "    :p.INSERT(('HAS_ID',        1,    NULL, NULL));\n"
    "    :p.INSERT(('FUNCTIONALITY', 0,    NULL, NULL));\n"
    "    d = SELECT * FROM \"SOAC_SF_ONE\";\n"
    "    m = SELECT * FROM \"SOAC_SF_EMPTY_MODEL\";\n"
    "    CALL _SYS_AFL.PAL_MLP_MULTI_TASK_PREDICT(:d, :p, :m, result, placeholder);\n"
    "END"
)
sql = sql.replace("{STATE_ID}", STATE_ID)

t0 = time.time()
cursor = myconn.connection.cursor()
try:
    cursor.execute(sql)
finally:
    cursor.close()
sf_ms = (time.time() - t0) * 1000

myconn.drop_table("SOAC_SF_ONE")
myconn.drop_table("SOAC_SF_EMPTY_MODEL")
print(f"Stateful predict  (1 order): {sf_ms:.0f} ms  — model already in memory")

In [ ]:
import pandas as pd

# Delete: remove model state from HANA server memory
clf_ft.delete_model_state()
print("Model state deleted from HANA server memory.\n")

# Speed comparison
speed = pd.DataFrame([
    {"Mode": "Standard predict", "Time (1 order)": f"{std_ms:.0f} ms", "Notes": "Reloads model from disk each call"},
    {"Mode": "Stateful predict",  "Time (1 order)": f"{sf_ms:.0f} ms",  "Notes": "Model stays in HANA server memory"},
])
display(speed.set_index("Mode"))
print(f"Speedup: {std_ms/sf_ms:.0f}x faster")

## A.2 - Load the SOAC Dataset into SAP HANA Cloud

The SOAC dataset comes from the [SAP/SALT dataset on Hugging Face](https://huggingface.co/datasets/SAP/SALT).
It was downloaded as **parquet files** — a file format for large tables, similar to Excel but much faster to read.

> **Note:** In the exercise, this data is already pre-loaded into `DA262_SHARED` for you.
> You only need this section if you are setting up the environment from scratch in your own HANA Cloud.

| File | Rows | What it is |
|---|---|---|
| `JoinedTables_train.parquet` | ~1,900,000 | Training data — the model learned from these rows |
| `JoinedTables_test.parquet` | ~402,855 | Test data — used to check how accurate the model is |

**What the three code cells below do:**
1. Read both files from your local computer into Python
2. Add a simple ID number to each row (1, 2, 3 …) — the parquet files do not have one
3. Upload both tables to SAP HANA Cloud — this takes about 15–20 minutes for the 1.9M training rows

> ⚠️ **Update `DATA_PATH`** in the first code cell to point to the folder where you saved the parquet files.

In [ ]:
import pandas as pd

# Path to the local parquet files — update this to your folder
DATA_PATH = r"<your data path>\SOAC original data"

df_train = pd.read_parquet(DATA_PATH + r"\JoinedTables_train.parquet")
df_test  = pd.read_parquet(DATA_PATH + r"\JoinedTables_test.parquet")

print("Training rows:", len(df_train))
print("Test rows    :", len(df_test))
print("Columns      :", df_train.columns.tolist())
df_train.head(3)

In [ ]:
# Add a simple integer ID column (1, 2, 3, ...) — the parquet files don't include one
df_train.insert(0, "ID", range(1, len(df_train) + 1))
df_test.insert(0,  "ID", range(1, len(df_test)  + 1))

print("Training shape:", df_train.shape)
print("Test shape    :", df_test.shape)

In [ ]:
import hana_ml.dataframe as dataframe

# Step 3: Upload both splits to SAP HANA Cloud
# Note: 1.9M training rows takes about 15-20 minutes to upload

dataframe.create_dataframe_from_pandas(myconn, df_train, "SOAC_TRAINING_DATA", schema=myuser, force=True)
print("Training data uploaded:", myconn.table("SOAC_TRAINING_DATA", schema=myuser).count(), "rows")

dataframe.create_dataframe_from_pandas(myconn, df_test, "SOAC_TESTDATA", schema=myuser, force=True)
print("Test data uploaded    :", myconn.table("SOAC_TESTDATA",        schema=myuser).count(), "rows")

## A.3 - Train the SOAC Base Full Model (Reference)

This is the SQL that was used to train the model you worked with in the exercise.
It was run **once** by the instructor — you do **not** need to run it again.

**What happened:**
The SQL was run with a SAP HANA Cloud instance, expaned with an **[Elastic Compute Node (ECN)](https://help.sap.com/docs/hana-cloud/sap-hana-cloud-administration-guide/create-and-manage-ecns-using-sap-hana-cloud-central)**.
Think of the ECN as temporarily resource extension computation node (here of ~256 GB memory)
just for the training job, which took about 2.5 hours.

| What | Plain English |
|---|---|
| Algorithm | `PAL_MLP_MULTI_TASK` — a neural network that predicts 8 fields at once |
| Training data | 1,916,685 sales orders (1.9 million rows, 27 fields each) |
| Network layers | 4 deep layers, each field represented by 32 numbers |
| Batch size | 256 orders looked at per training step |
| Learning rate | 0.01 — how big a step the model takes when adjusting itself |
| Training time | ~2.5 hours on the ERN node |
| Result | Model saved to `DA262_SHARED.SOAC_PRETRAINED_MODEL` (14,864 rows of weights) |

> ⚠️ **Do not run Step 4 (the CALL statement) — the model is already trained.**
> The line `-- SET 'APPLICATIONCOMPONENT' = 'APP_ERN_WORKLOAD'` at the top must stay commented out
> (the `--` at the start means it is disabled). Removing `--` would route a job to the ERN node.

In [ ]:
-- MODEL ALREADY TRAINED — SET line no longer needed
-- SET 'APPLICATIONCOMPONENT' = 'APP_ERN_WORKLOAD';

-- =============================================================================
-- Full 1.9M training data set with 27-column 

-- =============================================================================
-- Step 1 : Output tables + verify training data exists
-- =============================================================================
-- MODEL ALREADY TRAINED — do not drop/recreate or the 2.5h training will repeat
-- DROP TABLE "<your schema>"."SOAC_MODEL_19M";    # referred to as SOAC_PRETRAINED_MODEL in the exercises above

-- CREATE COLUMN TABLE "<your schema>"."SOAC_MODEL_19M" (
--     "ROW_INDEX"     INTEGER,
--     "PART_INDEX"    INTEGER,
--     "MODEL_CONTENT" NVARCHAR(5000)
-- );

-- DROP TABLE "<your schema>"."SOAC_MODEL_19M_TRAIN_LOG";

-- CREATE COLUMN TABLE "<your schema>"."SOAC_MODEL_19M_TRAIN_LOG" (
--     "ITERATION" INTEGER,
--     "ERROR"     DOUBLE
-- );

-- Verify training data: should show ~1,916,685
SELECT COUNT(*) AS TOTAL_TRAINING_ROWS FROM "<your schema>"."SOAC_TRAINING_DATA";

-- Verify column count: should show 28 (ID + 27 feature/target cols)
SELECT COUNT(*) AS COL_COUNT
FROM TABLE_COLUMNS
WHERE SCHEMA_NAME = '<your schema>' AND TABLE_NAME = 'SOAC_TRAINING_DATA';


-- =============================================================================
-- Step 2 : PAL parameter table
-- DROP/CREATE → Ctrl+Enter each; select all INSERT lines below → Alt+X
-- =============================================================================
DROP TABLE "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS";

CREATE COLUMN TABLE "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"; (
    "PARAM_NAME"   NVARCHAR(256),
    "INT_VALUE"    INTEGER,
    "DOUBLE_VALUE" DOUBLE,
    "STRING_VALUE" NVARCHAR(1000)
);

-- Key model params for a high-accuracy model and performance
-- ENABLE_STREAMING=1 streams data row-by-row (prevents OOM with large and high cardinality training data, speeds up training time)
-- EMBEDDED_NUM=32, RESIDUAL_NUM=4 matches demo notebook 
-- NETWORK_TYPE=1, RESNET
-- TRAINING_STYLE=1 (stochastic) 
-- BATCH_SIZE=256 
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('NETWORK_TYPE',        1,                    CAST(NULL AS DOUBLE), CAST(NULL AS NVARCHAR(1000)));
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('EMBEDDED_NUM',        32,                   CAST(NULL AS DOUBLE), CAST(NULL AS NVARCHAR(1000)));
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('RESIDUAL_NUM',        4,                    CAST(NULL AS DOUBLE), CAST(NULL AS NVARCHAR(1000)));
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('MAX_ITERATION',       100,                  CAST(NULL AS DOUBLE), CAST(NULL AS NVARCHAR(1000)));
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('BATCH_SIZE',          256,                  CAST(NULL AS DOUBLE), CAST(NULL AS NVARCHAR(1000)));
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('LEARNING_RATE',       CAST(NULL AS INTEGER), 0.01,                CAST(NULL AS NVARCHAR(1000)));
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('TRAINING_PERCENTAGE', CAST(NULL AS INTEGER), 0.9,                 CAST(NULL AS NVARCHAR(1000)));
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('EARLY_STOP',          1,                    CAST(NULL AS DOUBLE), CAST(NULL AS NVARCHAR(1000)));
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('SAVE_BEST_MODEL',     1,                    CAST(NULL AS DOUBLE), CAST(NULL AS NVARCHAR(1000)));
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('SEED',                42,                   CAST(NULL AS DOUBLE), CAST(NULL AS NVARCHAR(1000)));
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('HAS_ID',              1,                    CAST(NULL AS DOUBLE), CAST(NULL AS NVARCHAR(1000)));

INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('ENABLE_STREAMING',    1,                    CAST(NULL AS DOUBLE), CAST(NULL AS NVARCHAR(1000)));
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('TRAINING_STYLE',      1,                    CAST(NULL AS DOUBLE), CAST(NULL AS NVARCHAR(1000)));
-- All 8 targets as OVERLAPPED_VARIABLE — exactly matches demo notebook (label=None)
-- No DEPENDENT_VARIABLE (demo uses overlapped_variable=OVERLAPPED, label=None)
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('OVERLAPPED_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'SALESOFFICE');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('OVERLAPPED_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'SALESGROUP');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('OVERLAPPED_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'CUSTOMERPAYMENTTERMS');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('OVERLAPPED_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'SHIPPINGCONDITION');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('OVERLAPPED_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'SHIPPINGPOINT');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('OVERLAPPED_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'PLANT');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('OVERLAPPED_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'HEADERINCOTERMSCLASSIFICATION');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('OVERLAPPED_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'ITEMINCOTERMSCLASSIFICATION');
-- Categorical: all 27 feature + target columns
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'SALESOFFICE');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'SALESGROUP');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'CUSTOMERPAYMENTTERMS');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'SHIPPINGCONDITION');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'SHIPPINGPOINT');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'PLANT');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'HEADERINCOTERMSCLASSIFICATION');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'ITEMINCOTERMSCLASSIFICATION');

INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'SALESDOCUMENTTYPE');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'SALESORGANIZATION');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'DISTRIBUTIONCHANNEL');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'BILLINGCOMPANYCODE');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'TRANSACTIONCURRENCY');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'SALESDOCUMENTITEMCATEGORY');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'PRODUCT');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'SOLDTOPARTY');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'SHIPTOPARTY');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'BILLTOPARTY');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'PAYERPARTY');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'COUNTRY_SOLDTOPARTY');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'REGION_SOLDTOPARTY');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'COUNTRY_SHIPTOPARTY');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'REGION_SHIPTOPARTY');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'COUNTRY_BILLTOPARTY');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'REGION_BILLTOPARTY');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'COUNTRY_PAYERPARTY');
INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS"  VALUES ('CATEGORICAL_VARIABLE', CAST(NULL AS INTEGER), CAST(NULL AS DOUBLE), 'REGION_PAYERPARTY');

-- Verify: should show 48
SELECT COUNT(*) AS PARAM_COUNT FROM "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS";


-- =============================================================================
-- Step 3 : Create stored procedure
-- 3a → DROP PROCEDURE line → Ctrl+Enter  (ignore "not found" error)
-- 3b → select ENTIRE CREATE PROCEDURE...END; block → Ctrl+Enter
-- =============================================================================
DROP PROCEDURE "<your schema>"."SOAC_MODEL_19M_TRAIN_PROC";

CREATE PROCEDURE "<your schema>"."SOAC_MODEL_19M_TRAIN_PROC"()
LANGUAGE SQLSCRIPT AS
BEGIN
    DECLARE row_count   INTEGER;
    DECLARE train_data  TABLE LIKE "<your schema>"."SOAC_TRAINING_DATA";
    DECLARE param_tbl   TABLE LIKE "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS";
    DECLARE pre_model   TABLE LIKE "<your schema>"."SOAC_MODEL_19M"
    DECLARE model_out   TABLE ("ROW_INDEX" INTEGER, "PART_INDEX" INTEGER, "MODEL_CONTENT" NVARCHAR(5000));
    DECLARE log_out     TABLE ("ITERATION" INTEGER, "ERROR" DOUBLE);
    DECLARE stat_out    TABLE ("STAT_NAME" NVARCHAR(256), "STAT_VALUE" NVARCHAR(1000));
    DECLARE opt_out     TABLE ("PARAM_NAME" NVARCHAR(256), "INT_VALUE" INTEGER, "DOUBLE_VALUE" DOUBLE, "STRING_VALUE" NVARCHAR(1000));
    DECLARE ph_out      TABLE ("KEY" NVARCHAR(1000), "OBJECT" NVARCHAR(1000), "VALUE" NVARCHAR(1000));

    SELECT COUNT(*) INTO row_count FROM "ML_RAYWCR"."SOAC_MODEL_FULL_27COL_TBL";

    IF :row_count = 0 THEN
        train_data = SELECT * FROM "<your schema>"."SOAC_TRAINING_DATA";
        param_tbl  = SELECT * FROM "<your schema>"."SOAC_MODEL_19M_TRAIN_PARAMS";
        pre_model  = SELECT * FROM "<your schema>"."SOAC_MODEL_19M" WHERE 1 = 0;

        CALL _SYS_AFL.PAL_MLP_MULTI_TASK(
            :train_data,
            :param_tbl,
            :pre_model,
            model_out,
            log_out,
            stat_out,
            opt_out,
            ph_out
        );

        INSERT INTO "<your schema>"."SOAC_MODEL_19M"  SELECT * FROM :model_out;
        INSERT INTO "<your schema>"."SOAC_MODEL_19M_TRAIN_LOG"  SELECT * FROM :log_out;
    END IF;
END;


-- =============================================================================
-- Step 4 : Run training  (cursor on this line → Ctrl+Enter)
-- First run: ~3-6 hours.  Re-run: instant if model already saved.
-- IMPORTANT: Run SET line at top of file first in this same DBeaver tab.
-- =============================================================================
CALL "<your schema>"."SOAC_MODEL_19M_TRAIN_PROC"();


-- =============================================================================
-- Step 5 : Check results — each SELECT → Ctrl+Enter
-- =============================================================================
SELECT COUNT(*) AS MODEL_ROWS FROM "<your schema>"."SOAC_MODEL_19M";

SELECT * FROM "<your schema>"."SOAC_MODEL_19M_TRAIN_LOG" ORDER BY "ITERATION";

SELECT
    MIN("ITERATION") AS FIRST_ITER,
    MAX("ITERATION") AS LAST_ITER,
    MAX(CASE WHEN "ITERATION" = (SELECT MIN("ITERATION") FROM "<your schema>"."SOAC_MODEL_19M_TRAIN_LOG"") THEN "ERROR" END) AS START_ERROR,
    MAX(CASE WHEN "ITERATION" = (SELECT MAX("ITERATION") FROM "<your schema>"."SOAC_MODEL_19M_TRAIN_LOG") THEN "ERROR" END) AS FINAL_ERROR
FROM "<your schema>"."SOAC_MODEL_19M_TRAIN_LOG";


<br>